# 00 · Config & common objects
Everything shared lives here. The other two notebooks start with `%run ./00_config` and then only describe **what** to load.

**Flow:** CSV files in a Unity Catalog Volume → `bronze` (raw text + lineage) → `silver` (typed, rules applied, latest record, duplicates removed) → `gold` (dimensions and facts)

| You write (in 01 / 02) | Helper (defined below) | Result |
|---|---|---|
| `load_silver("customers", {...columns...}, key="customer_id")` | `ingest` + `load_silver` | `bronze.customers`, `silver.customers`, `silver.customers_quarantine` |
| `load_dim("dim_customer", source="customers", ...)` | `load_dim` | `gold.dim_customer` with surrogate keys and the Unknown member (-1) |
| `load_fact("fact_sales", <a SQL query>, keys=[...])` | `load_fact` | `gold.fact_sales` |

Rules live in **`dq_rules.yml`** (DQX format). Fill in the settings in the next cell (catalog, the landing folder, the rules file), nothing else is needed here.

In [ ]:
# ───────────── SETTINGS ─────────────
# Each setting is a notebook widget. Interactively (Git folder) the defaults below are used, so you normally change nothing.
# A job (see jobs/medallion_job.yml) overrides them with job parameters of the same name, per environment.
import os, re

def _setting(name, default):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()

# Unity Catalog catalog; the bronze / silver / gold schemas are created inside it
CATALOG       = _setting("catalog", "main_lf")

# Volume folder that holds the source files. Underneath it: a  dims  folder (customers_*.csv, products_*.csv, crm_accounts_*.csv, ...)
# and a  facts  folder (sales_*.csv, returns_*.csv). Files may sit directly in those folders or in sub-folders (Batch1, Batch2, ...).
LANDING_ROOT  = _setting("landing_root", "/Volumes/main_lf/default/data/Landing")

# Rules file. Default: dq_rules.yml next to this notebook (works in a Git folder and in a deployed bundle)
RULES_PATH    = _setting("rules_path", os.path.join(os.getcwd(), "dq_rules.yml"))

# Put in front of the schema names so developers and test runs get their own copies: "" -> bronze, "dev_anna_" -> dev_anna_bronze
# Interactive default = your own prefix (dev_<your user name>_), so working in a Git folder never touches the real schemas.
# The production job passes "" explicitly. To run interactively on the real schemas, set the widget schema_prefix to empty.
_me = re.sub(r"[^a-z0-9]+", "_", spark.sql("SELECT current_user()").first()[0].split("@")[0].lower())
SCHEMA_PREFIX = _setting("schema_prefix", f"dev_{_me}_")

In [ ]:
import yaml
from pyspark.sql import functions as F, Window
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

BRONZE, SILVER, GOLD = (f"{CATALOG}.{SCHEMA_PREFIX}{layer}" for layer in ("bronze", "silver", "gold"))
for value, label in ((LANDING_ROOT, "LANDING_ROOT"), (RULES_PATH, "RULES_PATH")):
    assert value, f"setting {label} is empty (SETTINGS cell of 00_config, or a job parameter)"
LANDING_DIMS, LANDING_FACTS = f"{LANDING_ROOT}/dims", f"{LANDING_ROOT}/facts"
LANDING = LANDING_DIMS                                # 01_dimensions reads from here; 02_facts switches this to LANDING_FACTS
for schema in (BRONZE, SILVER, GOLD):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# data-quality rules: read the YAML and let DQX validate every rule
dq = DQEngine(WorkspaceClient())
RULES = yaml.safe_load(open(RULES_PATH))
for source, rule in RULES.items():
    status = DQEngine.validate_checks(rule["checks"])
    assert not status.has_errors, f"invalid rules for {source}: {status.errors}"
print(f"rules: {sum(len(r['checks']) for r in RULES.values())} checks for {len(RULES)} sources | catalog {CATALOG} | schemas {SCHEMA_PREFIX}bronze/silver/gold | landing {LANDING_ROOT}")

## Bronze: files in the Volume → `bronze.<name>`
`ingest` lists the CSV files of one source in the landing folder (and its sub-folders) and loads only the files whose path is **not already in Bronze** (`_source_file`). Each new file is read with its **own header**, so a file with a new column, or with the columns in a different order, is handled correctly. Drop `customers_batch2.csv` next to `customers_batch1.csv`, re-run, and only the new file is read and added. There are no checkpoints or streams to manage. Files are matched by name: `customers_*.csv` feeds `bronze.customers`.

Bronze keeps every column as text and adds `_source_file` (the full file path) and `_ingest_ts`.

In [ ]:
import re, fnmatch
from functools import reduce

def _list_csv(folder, name):
    """Paths of all files <name>_*.csv in the folder and its sub-folders."""
    found, todo = [], [folder]
    while todo:
        for f in dbutils.fs.ls(todo.pop()):
            if f.name.endswith("/"): todo.append(f.path)
            elif fnmatch.fnmatch(f.name, f"{name}_*.csv"): found.append(f.path)
    return sorted(found)

_NO_SCHEME = r"^[A-Za-z][A-Za-z0-9+.\-]*:"          # 'dbfs:/Volumes/x' and '/Volumes/x' are the same file

def ingest(name):
    """Landing files <name>_*.csv -> bronze.<name>. Reads and appends only files that are not in bronze yet (judged by file path)."""
    target = f"{BRONZE}.{name}"
    files = _list_csv(LANDING, name)
    if not files:
        print(f"   no {name}_*.csv files found in {LANDING}"); return
    if spark.catalog.tableExists(target):
        loaded = {r[0] for r in spark.table(target).select(F.regexp_replace("_source_file", _NO_SCHEME, "")).distinct().collect()}
        files = [p for p in files if re.sub(_NO_SCHEME, "", p) not in loaded]       # skip files loaded before
    for i in range(0, len(files), 50):                                              # 50 files per write
        parts = [spark.read.option("header", "true").csv(p)                         # every column as text, the file's own header
                      .select("*", F.col("_metadata.file_path").alias("_source_file"), F.current_timestamp().alias("_ingest_ts"))
                 for p in files[i:i + 50]]
        new = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), parts)
        new.write.mode("append").option("mergeSchema", "true").saveAsTable(target)

## Silver: `bronze.<name>` → `silver.<name>`
`load_silver(name, columns, key)` does the whole step for one source, and is **incremental**: only files that Silver has not processed yet go through typing and the data-quality rules.
1. loads new files into Bronze (`ingest`)
2. takes only the Bronze rows of files not processed before, casts each column to its type (`"string:lower"`, `"decimal(12,2)"`, `"timestamp"`...; blank text becomes NULL, a value that does not fit becomes NULL) and applies the DQX rules from `dq_rules.yml`
3. rows that fail an **error** rule are added to a store of rejected rows (`silver._failed_<name>`); **warn** rules only count
4. valid rows are compared with the latest valid record per `key` kept so far (`silver._latest_<name>`); the newer one wins (`updated_at`, then arrival time)
5. **de-duplication runs over the whole set** (every latest record, not only the new ones), if the rules file has a `dedupe` block for the source: first submitted wins, the others go to quarantine. A late record can therefore replace an earlier "first" one
6. writes `silver.<name>`, `silver.<name>_quarantine` (rejected rows + duplicates) and a summary of every rule to `silver.dq_log`
7. remembers the processed files (`silver._files_<name>`)

Every call also adds **one row to `silver.load_log`**: when it ran, `loaded`, `nothing new` or `failed` (with the error text), how many new files and rows it processed, how many of those were rejected, the totals now in `silver.<name>` and its quarantine, and the seconds it took.

The result is the same as rebuilding Silver from all of Bronze, but the rules only run on new data. Re-running with no new file does nothing. After you change a rule or the columns of a source, call `load_silver(..., rebuild=True)` once: it forgets what was processed and rebuilds that source from all of Bronze.

In [ ]:
TRANSFORMS = {"lower": "lower({})", "upper": "upper({})", "digits": "nullif(right(regexp_replace({}, '[^0-9]', ''), 10), '')"}

def _typed(column, spec):
    """'string:lower' / 'bigint' / 'decimal(12,2)' -> a trimmed, blank-to-NULL, cast column."""
    kind, _, transform = spec.partition(":")
    expr = f"nullif(trim(`{column}`), '')"
    if kind != "string":
        expr = f"try_cast({expr} AS {kind})"           # a value that does not fit becomes NULL
    if transform:
        expr = TRANSFORMS[transform].format(expr)
    return F.expr(expr).alias(column)

def _rule_names(column):
    return F.transform(column, lambda e: e["name"])     # DQX result structs -> just the rule names

def _step(label, since):
    import time
    now = time.time(); print(f"   {label:<26s} {now - since:5.1f}s"); return now

def _overwrite(df, target):
    """Replace a table with df, also when df was read from that same table."""
    tmp = f"{target}__tmp"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tmp)
    spark.table(tmp).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    spark.sql(f"DROP TABLE {tmp}")

def _load_silver(name, columns, key, order_by, rebuild):
    """the work of load_silver; returns what happened, for silver.load_log"""
    import time
    keys = [key] if isinstance(key, str) else key
    rules = RULES[name]
    latest_t, failed_t, files_t = f"{SILVER}._latest_{name}", f"{SILVER}._failed_{name}", f"{SILVER}._files_{name}"
    print(f"silver.{name}")
    t = time.time()

    # 1. new files -> bronze
    ingest(name)
    t = _step("1 bronze (new files)", t)

    if rebuild:                                                   # forget what was processed: all of bronze is processed again
        for table in (latest_t, failed_t, files_t):
            spark.sql(f"DROP TABLE IF EXISTS {table}")
    first_run = not spark.catalog.tableExists(latest_t)

    # which bronze rows are new for silver = rows of files not processed before
    new = spark.table(f"{BRONZE}.{name}")
    if spark.catalog.tableExists(files_t):
        new = new.join(spark.table(files_t), "_source_file", "left_anti")
    new_files = [r[0] for r in new.select("_source_file").distinct().collect()]
    if not new_files and not first_run:
        print("   nothing new"); return {"status": "nothing new"}

    # 2. type the NEW rows and run the rules ONCE, and save the result as a work table
    #    (Spark would otherwise repeat this whole calculation for every output below)
    typed = new.select(*[_typed(c, t_) for c, t_ in columns.items()], F.struct(*new.columns).alias("_raw"), "_ingest_ts")
    work = f"{SILVER}._work_{name}"
    dq.apply_checks_by_metadata(typed, rules["checks"]).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(work)
    checked = spark.table(work)
    new_rows, new_rejected = checked.count(), checked.filter("_errors IS NOT NULL").count()   # for silver.load_log
    t = _step(f"2 typed + rules ({len(new_files)} file(s))", t)

    # 3. rejected rows are added to the store (a file already in the store is skipped, so a repeated run adds nothing twice)
    failed = checked.filter("_errors IS NOT NULL").select("_raw.*", _rule_names("_errors").alias("_errors"))
    if spark.catalog.tableExists(failed_t):
        failed = failed.join(spark.table(failed_t).select("_source_file").distinct(), "_source_file", "left_anti")
    failed.write.mode("append").option("mergeSchema", "true").saveAsTable(failed_t)

    # 4. latest valid record per key = the newer of what we kept so far and the new valid rows
    valid = checked.filter("_errors IS NULL").drop("_errors", "_warnings")
    if spark.catalog.tableExists(latest_t):
        try:
            valid = spark.table(latest_t).unionByName(valid)
        except Exception as e:
            raise RuntimeError(f"silver '{name}': the stored records no longer match the new files ({e}). "
                               f"Run load_silver('{name}', ..., rebuild=True) once.")
    newest = Window.partitionBy(*keys).orderBy(F.desc(order_by), F.desc("_ingest_ts"))
    _overwrite(valid.withColumn("_rn", F.row_number().over(newest)).filter("_rn = 1").drop("_rn"), latest_t)
    t = _step("3 latest per key", t)

    # 5. de-duplication over the WHOLE set of latest records, not just the new ones
    latest = spark.table(latest_t)
    rejected = spark.table(failed_t)
    dedupe = rules.get("dedupe")
    if dedupe:                                                                 # same values in several columns = same record
        first = Window.partitionBy(*dedupe["keys"]).orderBy(order_by, *keys)
        complete = F.expr(" AND ".join(f"`{k}` IS NOT NULL" for k in dedupe["keys"]))
        latest = latest.withColumn("_dup", complete & (F.row_number().over(first) > 1))
        duplicates = latest.filter("_dup").select("_raw.*", F.array(F.lit(dedupe["name"])).alias("_errors"))
        rejected = rejected.unionByName(duplicates, allowMissingColumns=True)
        latest = latest.filter("NOT _dup")

    # 6. write silver + quarantine
    latest.select(*columns).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}")
    rejected.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.{name}_quarantine")
    t = _step("4 silver + quarantine", t)

    # 7. what happened, per rule (appended to silver.dq_log on every run; the counts are totals over all data so far)
    quarantine = spark.table(f"{SILVER}.{name}_quarantine")
    errors = quarantine.select(F.explode("_errors").alias("rule")).groupBy("rule").count().withColumn("severity", F.lit("error"))
    warnings = (checked.filter("_warnings IS NOT NULL").select(F.explode(_rule_names("_warnings")).alias("rule"))
                       .groupBy("rule").count())
    log_t = f"{SILVER}.dq_log"
    if not first_run and spark.catalog.tableExists(log_t):                     # warnings of earlier runs are added to this run's
        warned = spark.table(log_t).filter(f"source = '{name}' AND severity = 'warn'")
        last = warned.agg(F.max("run_ts")).first()[0]
        if last is not None:
            before = warned.filter(F.col("run_ts") == F.lit(last)).select("rule", F.col("rows").alias("count"))
            warnings = warnings.unionByName(before).groupBy("rule").agg(F.sum("count").alias("count"))
    log = errors.unionByName(warnings.withColumn("severity", F.lit("warn")))
    (log.select(F.lit(name).alias("source"), "severity", "rule", F.col("count").cast("long").alias("rows"), F.current_timestamp().alias("run_ts"))
        .write.mode("append").saveAsTable(log_t))
    spark.sql(f"DROP TABLE IF EXISTS {work}")
    t = _step("5 rule log", t)

    # 8. remember the processed files (last, so an interrupted run is simply repeated)
    spark.createDataFrame([(f,) for f in new_files], "_source_file string").write.mode("append").saveAsTable(files_t)
    good, bad = spark.table(f"{SILVER}.{name}").count(), quarantine.count()
    print(f"   -> {good} rows loaded | {bad} in quarantine")
    return {"status": "loaded", "new_files": len(new_files), "new_rows": new_rows, "new_rejected": new_rejected}

def _log_load(name, status, started, new_files=0, new_rows=0, new_rejected=0, error=None):
    """one row in silver.load_log for every load_silver call, also when nothing was new or the load failed"""
    import time
    count = lambda t: spark.table(t).count() if spark.catalog.tableExists(t) else None
    row = (name, status, new_files, new_rows, new_rejected, count(f"{SILVER}.{name}"), count(f"{SILVER}.{name}_quarantine"),
           round(time.time() - started, 1), error)
    schema = ("source string, status string, new_files long, new_rows long, new_rejected long, "
              "silver_rows long, quarantine_rows long, seconds double, error string")
    (spark.createDataFrame([row], schema).withColumn("run_ts", F.current_timestamp())
          .select("run_ts", "source", "status", "new_files", "new_rows", "new_rejected", "silver_rows", "quarantine_rows", "seconds", "error")
          .write.mode("append").option("mergeSchema", "true").saveAsTable(f"{SILVER}.load_log"))

def load_silver(name, columns, key, order_by="updated_at", rebuild=False):
    """bronze.<name> -> silver.<name> (incremental). Writes one row per call to silver.load_log: loaded / nothing new / failed."""
    import time
    started = time.time()
    try:
        result = _load_silver(name, columns, key, order_by, rebuild)
    except Exception as e:
        try: _log_load(name, "failed", started, error=str(e)[:500])
        except Exception: pass                                   # never hide the real error behind a logging problem
        raise
    _log_load(name, started=started, **result)

## Gold: dimensions and facts
**`load_dim`** copies the chosen columns from a Silver table into `gold.<dim>`. A new business key gets the next surrogate key, a changed row is updated, a row with the key `-1` is the **Unknown member** that facts use when a dimension row is not there (yet).

**`load_fact`** takes a query (written by you in SQL, so the joins are visible) and inserts new rows, updates changed ones and **deletes rows that are no longer in the result** in `gold.<fact>` (for example a sales line that Silver has since rejected as a duplicate). Because it is rebuilt from Silver and the dimensions every run, a fact row that pointed at the Unknown member is **fixed automatically** when its dimension row arrives, and a fact follows a change in any table it joins.

**`load_dim_scd2`** is the same idea for a dimension that keeps **history** (Type 2): when a column listed in `track` changes, the current row is closed (`effective_to`, `is_current = false`) and a new row with a new surrogate key is added. Use it only where history matters; `01_dimensions` uses it for `dim_customer_360`.

All three use `upsert` (a Delta `MERGE` that only touches rows that changed; for facts it also deletes with `WHEN NOT MATCHED BY SOURCE`).

In [ ]:
def upsert(df, target, keys, delete_missing=False):
    """Insert new rows and update changed rows (Delta MERGE). Creates the table on the first run.
    delete_missing=True also deletes target rows that are no longer in df (used for facts, which are rebuilt in full)."""
    if not spark.catalog.tableExists(target):
        df.write.saveAsTable(target)
        return
    df.createOrReplaceTempView("_upsert_source")
    on = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    changed = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in df.columns if c not in keys)
    spark.sql(f"""MERGE INTO {target} t USING _upsert_source s ON {on}
                  WHEN MATCHED AND ({changed}) THEN UPDATE SET *
                  WHEN NOT MATCHED THEN INSERT *
                  {"WHEN NOT MATCHED BY SOURCE THEN DELETE" if delete_missing else ""}""")

def load_dim(name, source, key, sk, attrs, derived=None):
    """silver.<source> -> gold.<name>.  attrs = columns to copy, derived = {new_column: "SQL expression"}."""
    target = f"{GOLD}.{name}"
    src = spark.table(f"{SILVER}.{source}").select(key, *attrs, *[F.expr(e).alias(c) for c, e in (derived or {}).items()])
    if spark.catalog.tableExists(target):
        current = spark.table(target).select(key, sk)
        last = max(current.agg(F.max(sk)).first()[0] or 0, 0)
        existing = src.join(current, key)                                        # known keys keep their surrogate key
        new = src.join(current, key, "left_anti").withColumn(sk, F.lit(last) + F.row_number().over(Window.orderBy(key)))
        rows = existing.unionByName(new)
    else:
        rows = src.withColumn(sk, F.row_number().over(Window.orderBy(key)))
    unknown = spark.range(1).select(F.lit(-1).cast("bigint").alias(sk), F.lit(-1).cast(src.schema[key].dataType).alias(key),
                                    *[F.lit(None).cast(f.dataType).alias(f.name) for f in src.schema if f.name != key])
    upsert(rows.unionByName(unknown).select(sk, key, *[f.name for f in src.schema if f.name != key]), target, [key])
    print(f"gold.{name:18s} {spark.table(target).count():5d} rows (incl. the Unknown member)")

def load_dim_scd2(name, source, key, sk, attrs, track, derived=None):
    """Type 2 dimension: a change in a `track` column closes the current row and adds a new version.
    Every other attribute is simply overwritten in the current row (Type 1).
    Extra columns: effective_from, effective_to (9999-12-31 while open), is_current. Each version has its own surrogate key."""
    target = f"{GOLD}.{name}"
    now = F.lit(spark.sql("SELECT current_timestamp()").first()[0])               # one timestamp for the whole run
    far = F.lit("9999-12-31 00:00:00").cast("timestamp")
    src = spark.table(f"{SILVER}.{source}").select(key, *attrs, *[F.expr(e).alias(c) for c, e in (derived or {}).items()])
    cols = [f.name for f in src.schema if f.name != key]
    order = [sk, key, *cols, "effective_from", "effective_to", "is_current"]
    fingerprint = lambda d: F.sha2(F.concat_ws("||", *[F.coalesce(d[c].cast("string"), F.lit("~null~")) for c in track]), 256)

    if not spark.catalog.tableExists(target):                                    # first run: every row is version 1
        rows = (src.withColumn(sk, F.row_number().over(Window.orderBy(key))).withColumn("effective_from", now)
                   .withColumn("effective_to", far).withColumn("is_current", F.lit(True)))
        unknown = spark.range(1).select(F.lit(-1).cast("bigint").alias(sk), F.lit(-1).cast(src.schema[key].dataType).alias(key),
                                        *[F.lit(None).cast(src.schema[c].dataType).alias(c) for c in cols],
                                        F.lit("1900-01-01").cast("timestamp").alias("effective_from"), far.alias("effective_to"), F.lit(True).alias("is_current"))
        upsert(rows.select(*order).unionByName(unknown.select(*order)), target, [sk])
    else:
        current = spark.table(target).filter(f"is_current AND {sk} <> -1")
        old = current.select(key, F.col(sk).alias("_old_sk"), fingerprint(current).alias("_old_fp"))
        m = src.withColumn("_fp", fingerprint(src)).join(old, key, "left")
        new_version = m.filter("_old_sk IS NULL OR _fp <> _old_fp")                # a new person, or a tracked column changed
        unchanged   = m.filter("_old_sk IS NOT NULL AND _fp = _old_fp")

        closed  = (current.join(new_version.filter("_old_sk IS NOT NULL").select(key), key)       # the old version ends now
                          .withColumn("effective_to", now).withColumn("is_current", F.lit(False)).select(*order))
        refresh = current.select(sk, key, "effective_from", "effective_to", "is_current").join(unchanged.select(key, *cols), key).select(*order)
        last = spark.table(target).agg(F.max(sk)).first()[0] or 0
        created = (new_version.select(key, *cols).withColumn(sk, F.lit(last) + F.row_number().over(Window.orderBy(key)))
                              .withColumn("effective_from", now).withColumn("effective_to", far).withColumn("is_current", F.lit(True)).select(*order))
        upsert(closed.unionByName(refresh).unionByName(created), target, [sk])

    t = spark.table(target)
    print(f"gold.{name:18s} {t.count():5d} rows = {t.filter('is_current').count()} current + {t.filter('NOT is_current').count()} history (incl. the Unknown member)")

def load_fact(name, df, keys):
    """Insert new fact rows, update changed ones and delete the ones that are gone. Safe to re-run."""
    upsert(df, f"{GOLD}.{name}", keys, delete_missing=True)
    print(f"gold.{name:18s} {spark.table(f'{GOLD}.{name}').count():5d} rows")

## Reports and start-over
`dq_report()` shows what every rule found in the latest run of each source. `load_report()` shows the history of the loads from `silver.load_log` (newest first; `load_report("sales")` for one source). `reset_all()` drops bronze, silver and gold, so the next run reloads every file (run it from a cell when you want to start from scratch; it never touches the landing files and is never called automatically).

In [ ]:
def dq_report():
    log = spark.table(f"{SILVER}.dq_log")
    latest = log.groupBy("source").agg(F.max("run_ts").alias("run_ts"))
    return (log.join(latest, ["source", "run_ts"]).select("source", "severity", "rule", "rows")
               .orderBy("source", "severity", "rule"))

def load_report(source=None, runs=30):
    """the last <runs> loads from silver.load_log, newest first"""
    log = spark.table(f"{SILVER}.load_log")
    if source:
        log = log.filter(F.col("source") == source)
    return log.orderBy(F.desc("run_ts"), "source").limit(runs)

def reset_all():
    for schema in (BRONZE, SILVER, GOLD):
        spark.sql(f"DROP SCHEMA IF EXISTS {schema} CASCADE")
        spark.sql(f"CREATE SCHEMA {schema}")
    print("bronze, silver and gold were reset; the next run reloads every file")